In [23]:
from collections import defaultdict
import logging
import os
import sys
import yaml

from hydra import initialize, compose
from matplotlib.patches import PathPatch, Rectangle
import matplotlib.pyplot as plt
import numpy as np
from omegaconf import OmegaConf
import pandas as pd
import scanpy as sc
from scipy.special import softmax
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import LabelEncoder
import torch
from tqdm import tqdm

from labcompass.constants import ParamsFields
from labcompass.metrics import compute_e_distance


In [2]:
logger = logging.getLogger("notebook")
logger.setLevel(logging.INFO)

# Avoid adding handlers multiple times when cells re-run
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(message)s",
        datefmt="%H:%M:%S"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.propagate = False

In [3]:
BASE_DIR = "../../../.."
sys.path.insert(0, f"{BASE_DIR}/model_utils")
sys.path.insert(0, f"{BASE_DIR}/inverse/experiments")
from experiment_utils import get_forward_model, create_dir
from z_norm_modules import ZNorm, IZNorm, RescaledTargetPredictionModel



In [4]:
ANNOTATION_CONFIG_PATH = "../../../../generative_modeling/conditional_flow_matching/config/annotation/default.yaml"
RESULTS_DIR = "../../../../project_folder/results"
CONFIG_DIR = "../config"
# PLOTS_DIR = "../../../../project_folder/plots/plots_presentation"
PLOTS_DIR = "../../../../project_folder/plots_presentation"
CONFIG_NAME = "run_inverse"
cell_state_rep = "X_channel_standardized+X_scatter_standardized"
condition_rep = "condition_concat"

In [5]:
with initialize(config_path=CONFIG_DIR, version_base=None):
    base_cfg = compose(config_name=CONFIG_NAME)
print(base_cfg)

{'forward_model': {'num_time_steps': 1000, 'solver_kwargs': {'method': 'euler'}}, 'loss_guidance': {'fix_noise': True, 'num_forward_pass_per_sample': 350, 'regularization': None, 'reg_strength': 0.0, 'n_time_steps_forward_model': 50, 'solver_kwargs_forward_model': {'method': 'euler'}}, 'non_linearity': {'non_linearity_id': 'relu', 'non_linearity_kwargs': None}, 'paths': {'perturbation_prediction_path': '/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/dumps/2025-12-14_11-53-04/dainty-fog-107_FlowMatching.pkl', 'ct_classifier_path': '/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/dumps/2025-11-14_21-22-45/dashing-frog-1258_TargetPredictionModel.pkl', 'prior_flow_path': '/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/prior_flow_FlowMatching.pkl', 'dump_dir': '/lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/results'}, 'run': {'random_seed': 42}, 'sampling': {'target_cell_type': 'HSCs', 'N': 125, 'num_time

In [ ]:
def get_rescaling(
    adata: sc.AnnData,
    channel_params_key: str = "X_channel_params", 
    scatter_params_key: str = "X_scatter_params",
    inverse: bool = False,
) -> ZNorm | IZNorm:
    X_channel_params_fwd = adata.uns[channel_params_key]
    X_scatter_params_fwd = adata.uns[scatter_params_key]
    params = {
        ParamsFields.MEAN: torch.from_numpy(
            np.concatenate(
                (
                    X_channel_params_fwd[ParamsFields.MEAN],
                    X_scatter_params_fwd[ParamsFields.MEAN]
                ), axis=0
            )
        ),
        ParamsFields.COVARIANCE: torch.from_numpy(
            np.concatenate(
                (
                    X_channel_params_fwd["std"],
                    X_scatter_params_fwd["std"]
                ), axis=0
            )
        ),
    }
    if inverse:
        return IZNorm(params)
    return ZNorm(params)

def get_distance_df(X_pred, classes, adata_g, obsm_key="rescaled_features", fn=compute_e_distance, take_mean=True):
    edist_dict = defaultdict(list)
    for ct in tqdm(classes):
        x_ct = adata_g[adata_g.obs.cell_type == ct].obsm[obsm_key]
        if take_mean:
            x_ct = x_ct.mean(0, keepdims=True)
        edist = fn(X_pred, x_ct)
        # edist_dict["ct"].append(ct)
        edist_dict[ct].append(edist)
    df = pd.DataFrame(edist_dict)#.sort_values("edist", axis=0, ascending=False).T
    return df

In [7]:
with open(ANNOTATION_CONFIG_PATH, "r") as fb:
    annotation_dict = yaml.safe_load(fb)

In [8]:
forward_model, (
    perturbation_response_prediction_model,
    target_prediction_model,
) = get_forward_model(base_cfg, logger=logger)

13:35:41 | INFO | Loading Perturbation Response Prediction Model From /lustre/groups/ml01/workspace/lorenzo.consoli/projects/SFC_cambridge/dumps/2025-12-14_11-53-04/dainty-fog-107_FlowMatching.pkl...
13:39:00 | INFO | Model loaded!
NeuralVelocityField(
  (vf_modules): ModuleDict(
    (x_encoder): MLPBlock(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=27, out_features=128, bias=True)
          (1): ELU(alpha=1.0)
        )
        (1): Sequential(
          (0): Linear(in_features=128, out_features=128, bias=True)
          (1): ELU(alpha=1.0)
        )
        (2): Sequential(
          (0): Linear(in_features=128, out_features=128, bias=True)
          (1): Identity()
        )
      )
    )
    (time_encoder): MLPBlock(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=1024, out_features=512, bias=True)
          (1): ELU(alpha=1.0)
        )
        (1): Sequential(
          (0): Linear(in_features=512, out_fe

In [9]:
train_adata_phi = perturbation_response_prediction_model.train_data.adata
val_adata_phi = perturbation_response_prediction_model.validation_data[0].adata
print(f"{train_adata_phi=}\n{val_adata_phi=}")

train_adata_g = target_prediction_model.train_data.adata
val_adata_g = target_prediction_model.validation_data.adata
print(f"{train_adata_g=}\n{val_adata_g=}")


train_adata_phi=AnnData object with n_obs × n_vars = 92953367 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'protocol_id'
    var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
    uns: 'meta', 'processing_steps', 'one_hot', '740-yp_[µm]_one_hot', 'mtg_[µm]_one_hot', 'rhflt3l_[ng_ml]_one_hot', 'gm-csf_[ng_ml]_one_hot', 'tpo_[ng_ml]_

In [10]:
n_train_cells = 70_000
n_val_cells = 30_000
train_adata_phi = sc.pp.sample(
    train_adata_phi,
    n=n_train_cells,
    copy=True
)
val_adata_phi = sc.pp.sample(
    val_adata_phi,
    n=n_val_cells,
    copy=True
)
print(f"{train_adata_phi=}, {val_adata_phi=}")

train_adata_phi=AnnData object with n_obs × n_vars = 70000 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'protocol_id'
    var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
    uns: 'meta', 'processing_steps', 'one_hot', '740-yp_[µm]_one_hot', 'mtg_[µm]_one_hot', 'rhflt3l_[ng_ml]_one_hot', 'gm-csf_[ng_ml]_one_hot', 'tpo_[ng_ml]_one

In [11]:
adata_phi = sc.concat((train_adata_phi, val_adata_phi), uns_merge="same")
sc.pp.pca(adata_phi)
sc.pp.neighbors(adata_phi)
sc.tl.umap(adata_phi)
adata_phi

/home/icb/lorenzo.consoli/miniconda3/envs/sc_exp_design/lib/python3.12/site-packages/numba/np/ufunc/parallel.py:373: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


AnnData object with n_obs × n_vars = 100000 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'protocol_id'
    uns: 'meta', 'processing_steps', 'one_hot', '740-yp_[µm]_one_hot', 'mtg_[µm]_one_hot', 'rhflt3l_[ng_ml]_one_hot', 'gm-csf_[ng_ml]_one_hot', 'tpo_[ng_ml]_one_hot', 'sr1_[nm]_one_hot', 'um171_[nm]_one_hot', 'um729_[µm]_one_hot', 'scf_[ng_ml]_one_hot', 'butyzamide_[nm]_one_h

In [12]:
adata_g = sc.concat((train_adata_g, val_adata_g), uns_merge="same")
adata_g

AnnData object with n_obs × n_vars = 100000 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type'
    uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap'
    obsm: 'X_pca', 'X_umap', 'X_scatter', 'X_scatter_standardized', 'X_channel', 'X_channel_standardized', 'X_channel+X_scatter', 'X_channel_standardized+X_scatter'

In [18]:
izn_g = get_rescaling(train_adata_g, inverse=True)
zn_phi = get_rescaling(train_adata_phi)
X_true = torch.from_numpy(adata_g.obsm[cell_state_rep]).float().cuda()
X_true = izn_g(X_true)
X_true = zn_phi(X_true).detach().cpu().numpy()
adata_g.obsm["rescaled_features"] = X_true
adata_g

AnnData object with n_obs × n_vars = 100000 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch', 'leiden', 'cell_type'
    uns: 'cell_type_colors', 'leiden', 'leiden_colors', 'meta', 'neighbors', 'processing_steps', 'umap'
    obsm: 'X_pca', 'X_umap', 'X_scatter', 'X_scatter_standardized', 'X_channel', 'X_channel_standardized', 'X_channel+X_scatter', 'X_channel_standardized+X_scatter'

In [13]:
# Prepare label encoder
ct_le = LabelEncoder()
ct_values = target_prediction_model.train_data.adata.obs["cell_type"].values
ct_le.fit(ct_values)
classes = ct_le.classes_.tolist()

In [82]:
ct_results_dict = {}
ct_list = os.listdir(RESULTS_DIR)
pbar = tqdm(range(len(ct_list)))
for ct in ct_list:
    ct_dir = os.path.join(RESULTS_DIR, ct)
    if not os.path.isdir(ct_dir):
        continue
    ct_runs =  os.listdir(ct_dir)
    runs_dict = {}
    pbar.set_description(f"Reading results for {ct}, {len(ct_runs)} runs found")
    pbar.update()
    for run in ct_runs:
        run_dir = os.path.join(ct_dir, run)
        if not os.path.isdir(run_dir):
            continue
        inverse_results = np.load(
            os.path.join(run_dir, "inverse_results.npz")
        )
        fwd_results = np.load(
            os.path.join(run_dir, "fwd_results.npz")
        )
        with open(
            os.path.join(run_dir, "config.yaml"), "r"
        ) as fp:
            run_config = yaml.safe_load(fp)
        runs_dict[run] = {
            "inverse_results": inverse_results,
            "fwd_results": fwd_results,
            "config": run_config,
        }
    ct_results_dict[ct] = runs_dict

Reading results for DCs-CD14, 73 runs found:  95%|█████████▌| 19/20 [03:40<00:12, 12.80s/it]           

In [96]:
        XDist_run = pd.concat(run_dfs, axis=0)


In [ ]:
plt.hist(XDist_run.loc[:, target_ct])

In [94]:
for target_ct, ct_runs_data in ct_results_dict.items():
    print(target_ct)
    for run_id, run_results in ct_runs_data.items():
        fwd_results = run_results["fwd_results"]
        X_gen = fwd_results["X"]
        X_channel_gen = fwd_results["X_channel"]
        X_scatter_gen = fwd_results["X_scatter"]
        ct_logits = fwd_results["ct_logits"]
        ct_probs = fwd_results["ct_probs"]
        ct_mean_probs = ct_probs.mean(1)

        run_dfs = []
        # state space evaluation
        for idx in range(X_gen.shape[0]):
            XDist_df = get_distance_df(
                X_gen[idx],
                classes,
                adata_g,
                take_mean=False
            )
            run_dfs.append(XDist_df)
        XDist_run = pd.concat(run_dfs, axis=0)
        break
    break

Early_Pro-B


(1766, 27)
(350, 27)


()
(6629, 27)
(350, 27)


()
(1200, 27)
(350, 27)
()
(185, 27)
(350, 27)
()
(505, 27)
(350, 27)
()
(658, 27)
(350, 27)
()
(6474, 27)
(350, 27)


()
(4403, 27)
(350, 27)


...

()
(5866, 27)
(350, 27)


()
(17135, 27)
(350, 27)


()
(222, 27)
(350, 27)
()
(454, 27)
(350, 27)
()
(13135, 27)
(350, 27)


 95%|█████████▍| 18/19 [00:34<00:01,  1.90s/it]


KeyboardInterrupt: 

In [92]:
XDist_df

,CD14-Mono,CD49c-Myeloid,CyclingProgenitor*,DCs-CD14,EarlyGMP,Early_Pro-B,EosBasoMastPre,EosBasoMastPre_2,EryPro,GMP-Cycle,GMP-Neutro,GMP-Neutro/CD16-Mono,HSCs,MEP,MLP,MPP,MgkPro,Pro-B,pDCs/cDCs
0,54.152826,29.420743,82.892737,617.224853,33.438323,133.966774,28.795074,11.342662,53.005242,16.486043,16.615535,114.604368,61.163808,18.276435,17.870467,9.693385,622.418987,90.372104,9.998211
